<a href="https://colab.research.google.com/github/chamarairesh1982/LearnPython/blob/main/Week8_of_Ensemble_Learning_Bagging_Boosting_Stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Learning Practical: Bagging, Boosting & Stacking

**Dataset:** Breast Cancer Wisconsin (built into scikit-learn — binary classification: malignant vs benign tumor)

**Goal:** Build intuition for *why* ensembles work by comparing:
- A single Decision Tree (baseline, high variance)
- **Bagging** → Random Forest
- **Boosting** → AdaBoost & Gradient Boosting (XGBoost)
- **Stacking** → combining diverse models with a meta-learner


## 0. Setup

In [ ]:
!pip install -q xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier,
                              StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

np.random.seed(42)
print('Libraries loaded.')

## 1. Load & Explore the Dataset

We use the **Breast Cancer Wisconsin (Diagnostic)** dataset: 569 samples, 30 numeric features (computed from digitized images of tumor cell nuclei), and a binary target (0 = malignant, 1 = benign).

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Shape:', X.shape)
print('Class balance:\n', y.value_counts())
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

## 2. Baseline: A Single Decision Tree

Before introducing ensembles, let's see the problem they solve. An unpruned decision tree tends to **overfit**: near-perfect training accuracy, but a noticeable drop on test data. This gap is the *variance* problem that bagging is designed to fix.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(X_train))
test_acc = accuracy_score(y_test, tree.predict(X_test))

print(f'Single Decision Tree -> Train accuracy: {train_acc:.3f} | Test accuracy: {test_acc:.3f}')
print(f'Gap (overfitting signal): {train_acc - test_acc:.3f}')

results = {}  # we'll collect all model results here for a final comparison
results['Decision Tree (baseline)'] = test_acc

## 3. Bagging: Reducing Variance

### 3.1 Manual Bagging with `BaggingClassifier`
We take the *same* unstable model (a decision tree) and train many copies on different bootstrap samples, then average their votes.

In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    max_samples=1.0,      # bootstrap sample size = 100% of training data (with replacement)
    bootstrap=True,
    oob_score=True,       # free validation estimate using left-out samples
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train, y_train)

bag_train_acc = accuracy_score(y_train, bagging.predict(X_train))
bag_test_acc = accuracy_score(y_test, bagging.predict(X_test))

print(f'Bagging (100 trees) -> Train: {bag_train_acc:.3f} | Test: {bag_test_acc:.3f}')
print(f'Out-of-Bag score (free validation estimate): {bagging.oob_score_:.3f}')
print(f'Gap shrank from {train_acc - test_acc:.3f} (single tree) to {bag_train_acc - bag_test_acc:.3f} (bagged)')

results['Bagging (100 trees)'] = bag_test_acc

### 3.2 Random Forest — Bagging + Random Feature Selection
Random Forest adds one more decorrelation trick: each split only considers a random subset of features. This usually improves on plain bagging.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_features='sqrt', oob_score=True,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))

print(f'Random Forest -> Test accuracy: {rf_test_acc:.3f} | OOB score: {rf.oob_score_:.3f}')
results['Random Forest'] = rf_test_acc

# Feature importance -- a nice bonus of tree-based ensembles
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
importances.plot(kind='barh', figsize=(6,4), title='Top 10 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 3.3 Effect of Number of Trees (Illustrating *Variance Reduction*)
Watch test accuracy stabilize as we add more independently-trained trees — this is bagging's variance reduction in action.

In [ ]:
tree_counts = [1, 5, 10, 25, 50, 100, 200]
accs = []
for n in tree_counts:
    m = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    accs.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(6,4))
plt.plot(tree_counts, accs, marker='o')
plt.xlabel('Number of Trees')
plt.ylabel('Test Accuracy')
plt.title('Bagging: Accuracy Stabilizes as Trees Are Added')
plt.grid(alpha=0.3)
plt.show()

## 4. Boosting: Reducing Bias

### 4.1 AdaBoost
AdaBoost starts with a very **weak** learner (a decision stump — 1 split) and sequentially reweights misclassified samples so each new stump focuses on the previous ones' mistakes.

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # deliberately weak learner
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_train, y_train)
ada_test_acc = accuracy_score(y_test, ada.predict(X_test))
print(f'AdaBoost -> Test accuracy: {ada_test_acc:.3f}')
results['AdaBoost'] = ada_test_acc

### 4.2 Gradient Boosting (scikit-learn) & XGBoost
Gradient boosting fits each new weak learner to the *residual errors* of the current ensemble, rather than reweighting samples. XGBoost is a highly optimized, regularized implementation of this idea.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=2, random_state=42)
gb.fit(X_train, y_train)
gb_test_acc = accuracy_score(y_test, gb.predict(X_test))
print(f'Gradient Boosting -> Test accuracy: {gb_test_acc:.3f}')
results['Gradient Boosting'] = gb_test_acc

xgb = XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=2,
                     eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)
xgb_test_acc = accuracy_score(y_test, xgb.predict(X_test))
print(f'XGBoost -> Test accuracy: {xgb_test_acc:.3f}')
results['XGBoost'] = xgb_test_acc

### 4.3 Watch Out: Boosting *Can* Overfit
Unlike bagging, adding more boosting rounds doesn't always help — after a point, training accuracy keeps climbing while test accuracy plateaus or drops. Let's visualize this with a high learning rate to exaggerate the effect.

In [ ]:
n_estimators_range = [1, 10, 25, 50, 100, 200, 400]
train_accs, test_accs = [], []

for n in n_estimators_range:
    m = GradientBoostingClassifier(n_estimators=n, learning_rate=0.3, max_depth=3, random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(6,4))
plt.plot(n_estimators_range, train_accs, marker='o', label='Train accuracy')
plt.plot(n_estimators_range, test_accs, marker='o', label='Test accuracy')
plt.xlabel('Number of Boosting Rounds')
plt.ylabel('Accuracy')
plt.title('Boosting: Overfitting Risk with High Learning Rate')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Stacking: Combining Different Model Families

We combine three *structurally different* base learners:
- **Logistic Regression** (linear decision boundary)
- **K-Nearest Neighbors** (distance-based, local)
- **Support Vector Machine** (margin-based)

...and let a **meta-model** (another Logistic Regression) learn how to best combine their predictions. `StackingClassifier` handles the out-of-fold prediction generation internally, avoiding leakage.

In [ ]:
base_learners = [
    ('logreg', LogisticRegression(max_iter=5000)),
    ('knn', KNeighborsClassifier(n_neighbors=7)),
    ('svm', SVC(probability=True, kernel='rbf', random_state=42)),
]

stack = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=5000),
    cv=5,              # 5-fold CV to generate out-of-fold predictions for the meta-model
    n_jobs=-1
)
stack.fit(X_train, y_train)
stack_test_acc = accuracy_score(y_test, stack.predict(X_test))
print(f'Stacking Ensemble -> Test accuracy: {stack_test_acc:.3f}')
results['Stacking (LogReg+KNN+SVM)'] = stack_test_acc

# Compare against each base learner alone, for context
for name, model in base_learners:
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f'  Base learner alone -- {name}: {acc:.3f}')
    results[f'Base: {name}'] = acc

## 6. Final Comparison

Let's put every model side by side. Notice: the single decision tree usually has the biggest train/test gap (variance problem), while the ensembles close that gap or push accuracy higher outright.

In [ ]:
results_df = pd.Series(results).sort_values(ascending=True)

plt.figure(figsize=(8,6))
colors = ['#888888' if 'Base' in k or 'baseline' in k else '#2b6cb0' for k in results_df.index]
results_df.plot(kind='barh', color=colors)
plt.xlabel('Test Accuracy')
plt.title('Model Comparison: Baseline vs Bagging vs Boosting vs Stacking')
plt.xlim(min(results_df.min() - 0.02, 0.85), 1.0)
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

results_df.sort_values(ascending=False).to_frame('Test Accuracy')

In [ ]:
# A closer look at the best model's confusion matrix
best_model_name = results_df.idxmax()
print('Best performing model:', best_model_name)

# Re-fit XGBoost (typically a strong performer) for a detailed report as an example
y_pred = xgb.predict(X_test)
print(classification_report(y_test, y_pred, target_names=data.target_names))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=data.target_names)
plt.title('XGBoost Confusion Matrix')
plt.show()

## 7. Exercises

1. **Vary `max_depth`** in the base `DecisionTreeClassifier` used for bagging. Does bagging still help if the base tree is already shallow (low variance)? Why or why not?
2. **Vary `learning_rate`** in `GradientBoostingClassifier` from 0.01 to 1.0. At what point does the model start overfitting faster?
3. **Swap the base learners** in the `StackingClassifier` (e.g., add a `RandomForestClassifier` as a base learner). Does the meta-model improve?
4. **Try a different meta-model** in stacking (e.g., a small decision tree instead of logistic regression). Does it change performance?
5. **Bias/variance challenge:** Using `cross_val_score`, compute train vs. test accuracy for each technique across 5 folds and plot the variance of scores across folds. Which technique has the most *stable* (lowest variance) test performance?
6. **Real-world framing:** If you were deploying a model where interpretability and speed mattered more than the last 1% of accuracy, which of these four approaches would you pick, and why?